# 05A. Master Classical ML Benchmark: 8 Models Comparison
### Sonobuoy Sound Detective: Offline Underwater Acoustic Event Detection
**Role in Project:** The Final Showdown — Benchmarking all 8 algorithms across 4 teammates side-by-side.

### Models Benchmarked:
1. **Person 1:** Logistic Regression (`01_person1_lr_and_knn`)
2. **Person 1:** K-Nearest Neighbors (`01_person1_lr_and_knn`)
3. **Person 2:** Support Vector Machine (RBF) (`02_person2_svm_and_naive_bayes`)
4. **Person 2:** Gaussian Naive Bayes (`02_person2_svm_and_naive_bayes`)
5. **Person 3:** Decision Tree (`03_person3_decision_tree_and_random_forest`)
6. **Person 3:** Random Forest (`03_person3_decision_tree_and_random_forest`)
7. **Person 4:** AdaBoost (`04_person4_adaboost_and_xgboost`)
8. **Person 4:** XGBoost (`04_person4_adaboost_and_xgboost`)

We evaluate all 8 models on the identical held-out test set and produce:
- **Master Comparison Table:** Test Accuracy, Macro Precision, Macro Recall, Macro F1, Multi-Class ROC-AUC, Inference Latency.
- **Side-by-Side Confusion Matrices:** Direct visual comparison across all 8 algorithms.
- **Unified Multi-Model ROC Curves:** All 8 models plotted on the same figure.
- **Latency vs Accuracy Tradeoff Scatter Plot.**
- **Winner Declaration:** Declaring the top candidate for ground-station deployment.


In [ ]:
import os
import sys
import time
import json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc
)

Path("plots").mkdir(parents=True, exist_ok=True)
Path("reports").mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
print("Final 8-model comparison suite loaded.")


## 1. Load Test Dataset and Standardized Features


In [ ]:
parq_path = Path("../data/features/features.parquet")
csv_path  = Path("../data/features/features.csv")
sel_path  = Path("../data/features/selected_features.json")

df = pd.read_parquet(parq_path) if parq_path.exists() else pd.read_csv(csv_path)

with open(sel_path) as f:
    raw_sel = json.load(f)
    feature_cols = raw_sel["selected_features"] if isinstance(raw_sel, dict) and "selected_features" in raw_sel else raw_sel

test_mask = df["split"] == "test" if "split" in df.columns else df.index.isin(df.sample(frac=0.15, random_state=RANDOM_SEED).index)
df_test = df[test_mask].copy()

classes = sorted(df["label"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}
y_test = df_test["label"].map(class_to_idx).values
X_test_raw = np.nan_to_num(df_test[feature_cols].values, nan=0.0)

print(f"Loaded held-out test set: {len(df_test)} samples across classes: {classes}")


## 2. Load All 8 Models from Teammate Folders


In [ ]:
model_registry_list = [
    ("Logistic Regression", Path("../01_person1_lr_and_knn/models"), "logistic_regression"),
    ("K-Nearest Neighbors", Path("../01_person1_lr_and_knn/models"), "knn"),
    ("Support Vector Machine", Path("../02_person2_svm_and_naive_bayes/models"), "svm_rbf"),
    ("Gaussian Naive Bayes", Path("../02_person2_svm_and_naive_bayes/models"), "naive_bayes"),
    ("Decision Tree", Path("../03_person3_decision_tree_and_random_forest/models"), "decision_tree"),
    ("Random Forest", Path("../03_person3_decision_tree_and_random_forest/models"), "random_forest"),
    ("AdaBoost", Path("../04_person4_adaboost_and_xgboost/models"), "adaboost"),
    ("XGBoost", Path("../04_person4_adaboost_and_xgboost/models"), "xgboost"),
]

loaded_models = {}
loaded_scalers = {}

for name, m_dir, slug in model_registry_list:
    m_file = m_dir / f"model_{slug}.joblib"
    s_file = m_dir / f"scaler_{slug}.joblib"
    
    if not m_file.exists():
        m_file = m_dir / "model.joblib"
        s_file = m_dir / "scaler.joblib"

    if m_file.exists():
        loaded_models[name] = joblib.load(m_file)
        loaded_scalers[name] = joblib.load(s_file) if s_file.exists() else None
        print(f"  [OK] Loaded: {name:<25} from {m_dir.parent.name}")
    else:
        print(f"  [MISSING] {name} not found in {m_dir}!")

print(f"\nSuccessfully loaded {len(loaded_models)} / 8 classical algorithms.")


## 3. Run Comprehensive 8-Model Benchmark on Held-Out Test Set


In [ ]:
records = []
predictions = {}
probabilities = {}

for name, model in loaded_models.items():
    scaler = loaded_scalers.get(name)
    X_test_scaled = scaler.transform(X_test_raw) if scaler else X_test_raw
    
    t_start = time.perf_counter()
    n_iters = 50
    for _ in range(n_iters):
        _ = model.predict(X_test_scaled[:1])
    latency_ms = ((time.perf_counter() - t_start) / n_iters) * 1000.0
    
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled) if hasattr(model, "predict_proba") else None
    
    predictions[name] = y_pred
    probabilities[name] = y_prob
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)
    
    auc_val = None
    if y_prob is not None:
        try:
            auc_val = roc_auc_score(y_test, y_prob, multi_class="ovr", average="macro")
        except Exception:
            auc_val = None
            
    records.append({
        "Model": name,
        "Test Accuracy": acc,
        "Macro Precision": prec,
        "Macro Recall": rec,
        "Macro F1": f1,
        "ROC-AUC (Macro)": auc_val if auc_val is not None else 0.0,
        "Latency (ms/segment)": latency_ms
    })

df_results = pd.DataFrame(records).sort_values("Macro F1", ascending=False).reset_index(drop=True)
df_results.to_csv("reports/master_8_models_comparison.csv", index=False)

print("="*85)
print("  MASTER 8-MODEL CLASSICAL ALGORITHM BENCHMARK TABLE")
print("="*85)
display(df_results.style.format({
    "Test Accuracy": "{:.2%}",
    "Macro Precision": "{:.2%}",
    "Macro Recall": "{:.2%}",
    "Macro F1": "{:.2%}",
    "ROC-AUC (Macro)": "{:.3f}",
    "Latency (ms/segment)": "{:.2f} ms"
}))


## 4. Side-by-Side Confusion Matrices (All 8 Algorithms)


In [ ]:
n_models = len(predictions)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=range(len(classes)))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm, nan=0.0)
    
    sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.savefig("plots/all_8_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Unified ROC Curves (All 8 Algorithms Comparison)


In [ ]:
plt.figure(figsize=(9, 6))
palette = sns.color_palette("tab10", len(probabilities))

for (name, y_prob), color in zip(probabilities.items(), palette):
    if y_prob is not None:
        all_fpr = np.linspace(0, 1, 100)
        mean_tpr = np.zeros_like(all_fpr)
        for i in range(len(classes)):
            y_bin = (y_test == i).astype(int)
            fpr, tpr, _ = roc_curve(y_bin, y_prob[:, i])
            mean_tpr += np.interp(all_fpr, fpr, tpr)
        mean_tpr /= len(classes)
        macro_auc = auc(all_fpr, mean_tpr)
        plt.plot(all_fpr, mean_tpr, lw=2, color=color, label=f"{name} (AUC = {macro_auc:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparison of All 8 Classical ML Models (Macro ROC Curves)", fontsize=12, fontweight='bold')
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plots/all_8_models_roc_curves.png", dpi=300, bbox_inches="tight")
plt.show()


## 6. Latency vs Macro-F1 Trade-off Analysis


In [ ]:
plt.figure(figsize=(9, 5))
for _, row in df_results.iterrows():
    plt.scatter(row['Latency (ms/segment)'], row['Macro F1'], s=160, alpha=0.85, label=row['Model'])
    plt.annotate(row['Model'], (row['Latency (ms/segment)'] + 0.05, row['Macro F1']), fontsize=9)

plt.xlabel("Inference Latency (ms / 4s segment) [Lower is Better]")
plt.ylabel("Macro F1-Score [Higher is Better]")
plt.title("Deployment Efficiency: Classification Quality vs Processing Speed", fontsize=11, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("plots/latency_vs_f1_tradeoff.png", dpi=300, bbox_inches="tight")
plt.show()


## 7. Official Winner Selection
The winning model among all 8 algorithms is selected for deployment in the Ground-Station Decision Support System.
